### Neshyba 2026


# Electron in a Coulombic potential

## What's in common with the *Particle-on-a-Sphere* problem

In this exercise, you'll be looking at the behavior of an electron in a Coulomb potential -- i.e., the Hydrogen atom. Fortunately, you've solved something very similar to this in the past, namely the *Particle-on-a-Sphere* problem, so first let's have a look at what's in common with that effort. The Schrödinger equation is written in its usual form

$$
H\psi=E\psi \ \ \ \ (1)
$$

where $H$ consists of a potential and a kinetic term,

$$
H = PE(x,y,z) + {-\hbar^2 \over 2 m} \nabla^2 \ \ \ \ (2)
$$

where the last term on the right represents the kinetic energy of our particle, in which $\nabla^2$ (called the *Laplacian Operator*) is understood to operate in three dimensions. In Cartesian coordinates, that's

$$
\nabla^2 = {\partial^2 \over \partial x^2} + {\partial^2 \over \partial y^2} + {\partial^2 \over \partial z^2} \ \ \ \ (3)
$$

which is what we'll be using here. 

## What's new

The only *formal* difference to the *Particle-on-a-Sphere problem* is the potential energy, which in atomic units can be written 

$$
PE = -Z/r \ \ \ \ (4)
$$

where $Z=1$ for a Hydrogen nucleus, $Z=2$ for a Helium nucleus, etc. An example is shown in Fig. 1.

<p style='text-align: center;'>
<img src="http://webspace.pugetsound.edu/facultypages/nesh/Notebook/CoulombSlice.png" height="400" width="500"/>
<strong>Figure 1</strong>. A slice through the Coulomb potential energy, with the first few quantum energy levels.
</p>

There are, however, some numerical considerations that are different. As for the discretization of the space, we'd like the number of steps (here, variable *nsteps*) to be big enough to resolve details of the eigenfunctions of $H$, but small enough to diagonalize in a reasonable amount of time. An additional constraint is that we'll need *nsteps* to be an even number -- otherwise, Python will try to evaluate $-Z/0$, and throw an error. It seems that a value of *nsteps=20* is a good compromise.

Secondly, because the electron is not stuck on the surface of a sphere, we'll need a much bigger simulation box. Details are provided below.

## The idea of this exercise
The idea of this exercise is that you can bascially copy a lot of what you did before for the *Particle-on-a-Sphere* CGI, with modifications here and there. So -- it might help to open up that CGI as you go through this. The cell prompts are meant to direct you to the relevant sections.

## Learning goals
The main learning goals of this exercise are:
1. I can construct and diagonalize a hamiltonian matrix representing (numerically) the electron in a hydrogen atom.
1. I can describe the shapes and degeneracies of the resulting eigenfunctions.

In [ ]:
import pint; from pint import UnitRegistry; AssignQuantity = UnitRegistry(system='atomic').Quantity
import numpy as np
import scipy.linalg as spla
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pchemlibrary as PL
%matplotlib inline

In [ ]:
# Quantum constants
hbar = AssignQuantity(1,'atomic_unit_of_time * hartree'); print(hbar)
h = hbar*2*np.pi; print(h)
m = AssignQuantity(1,'atomic_unit_of_mass'); print(m)

### Defining the Cartesian space we're working in
In the cell below, define your Cartesian space, using our (PchemLibrary) version of meshgrid. A discretization size on the order of 20 steps, and a box width of 10 bohr, should work. You'll also need to compute *dx*, *dy*, and *dz*, as you did for the *Particle-on-a-Sphere* problem.

In [ ]:
# Your code here


### Creating our 3d potential energy surface
In the cell below, define a potential function that takes as input arguments the Cartesian grids you created above, and returns the Coulomb potential defined in Eq. 4 in the Introduction. Something like

    def CoulombicPot(xgrid,ygrid,zgrid,Z):
        ...
        return -Z/r

In [ ]:
# Define a Coulomb potential function
# Your code here


# Calculate the potential energy using that function, with Z=1 (call it Vofr)
# Your code here


### Inspecting your result
In the cell below plots this potential energy if we march along a horizontal line (with $y=0$ and $z=0$) through our space. If all has gone well, this will look qualitatively like Fig. 1 in the Introduction.

In [ ]:
# Take a slice through V(r) (varying x, but with y=z=0)
# Your code here


### Creating the potential energy matrix
Use the cell below to convert your potential energy function into matrix form.

In [ ]:
# Construct the matrix representation of the potential energy
# Your code here


### Creating the kinetic energy matrix
In the cell below, use the function kron3 (the 3d equivalent of the 2d kronecker delta algorithm) to assemble the 3-d kinetic energy matrix.

In [ ]:
# Defining Kronecker delta function for three spatial dimensions
def kron3(X,Y,Z):
    return np.kron(X,np.kron(Y,Z))

# Create the three kinetic energy matrices
# Your code here


# Create identity matrices of the same size
# Your code here


# Use our new kronecker delta function to assemble the kinetic energy matrix over all three spatial dimensions
# Your code here


### Make the Hamiltonian and solve (diagonalize) it
Below, construct the Hamiltonian and use spla.eigh to diagonalize it. A time-saving device is to tell spla.eigh that we want only eigenstates between zero and $V$ in energy. With that activated, for a discretization of $20\times20\times20$, this will take less than a minute. 

In [ ]:
# Construct the Hamiltonian (call it H)
# Your code here


# Diagonalize -- but if we request bound states only (E<0), it's faster (takes < 1 minute if nsteps = 20)
Epsi,psi = spla.eigh(H, subset_by_value=[-np.inf, 0])

### Next, we visualize the energies
The code below plots energies of all the "bound" states we got (i.e., states with energy less than zero).

In [ ]:
# Compute energy differences
Epsi_above0 = Epsi-Epsi[0]

# Visualize the energies against the potential energy
f=plt.figure()
ax=f.add_subplot(111)
n = np.shape(Epsi)[0]
plt.plot(x,Vofr_along_x,'gray',linewidth=5,label='V(x,y,z) at y,z=0')
for i in range(n-1,-1,-1):
    thisEnergyDifference = round(Epsi_above0[i],2)
    thisEnergy = round(Epsi[i],2)
    plt.plot( [x[0],x[-1]], [Epsi[i],Epsi[i]], label=str(thisEnergy)+' (gs+'+str(thisEnergyDifference)+')')
plt.grid('True')
plt.xlabel('x')
plt.ylabel('Energy')

# Legend on the side
L=plt.legend(bbox_to_anchor=(1.05,1),loc=2,borderaxespad=0.)
box=ax.get_position()
ax.set_position([box.x0,box.y0,0.7*box.width,box.height])

### Pause for Analysis: the degeneracies of your numerical results
In the cell below, describe the degeneracies of your results. Because these are numerical results, we'll have to decide on some kind of tolerance that says "if energies are this close, I'll call them equal." So, let's say two states are degenerate if their energies lie within ~0.02 hartrees of one another. 

But the way, there is actually one state that's missing -- it's because this is a numerical diagonalization -- so the correct answer for the third energy level (gs+0.38 or so) is the degeneracy you see, plus one.

YOUR ANSWER HERE

### Next, we visualize the states themselves
As before, the cell below shows the all bound ($E<0$) eigenfunctions.

In [ ]:
# Decide how many states we want to look at
plot_until_index = np.shape(Epsi)[0]

# Loop over them
for thisindex in range(0,plot_until_index):
    
    # Extract this state
    thispsi = psi[:,thisindex]*1000
    thisEpsi = round(Epsi[thisindex],2)

    # Visualize it in 3d
    values = np.real(thispsi).flatten()
    isovalfactor = 0.4
    maxposval = np.max(values)*isovalfactor
    maxnegval = -np.max(-values)*isovalfactor
    print('psi ', thisindex)
    fig = go.Figure(data=go.Isosurface(
        x=xgrid.flatten(),
        y=ygrid.flatten(),
        z=zgrid.flatten(),
        value=values,
        isomin=maxnegval,
        isomax=maxposval,
        caps=dict(x_show=False, y_show=False, z_show=False)
    ))
    fig.show()

### Pause for Analysis
As mentioned above, an eigenfunction is missing from the highest-energy degenerate set. What angular momentum quantum number would you associate with the missing eigenfunction?

YOUR ANSWER HERE

### Refreshing and saving your code
1. Use the dropdown menu Kernel/Restart
2. Use the dropdown menu Cell/Run All Above
3. Under the "File" dropdown menu item in the upper left is a disk icon. Press it now to save your work (you can, do this at any time as you're working on an assignment, actually).

### Validating
This step will help ensure that you didn't miss something (although it's not a guarantee). Find the "Validate" button and press it. If there are any errors or warnings, fix them.

### Finishing up
Assuming all this has gone smoothly, carry out three more steps (but read this carefully before starting):
1. Close this notebook using the "File/Close and Halt" dropdown menu
1. Using the Assignments tab, submit this notebook
1. Press the Logout tab of the Home Page